# 05 · Feature Engineering Design

**Amaç:** Realtime-safe, PIT-correct feature setini tasarlamak ve smoke-testlemek. Asıl implementation `src/features/{instant.py, historical.py}` içindedir.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
sns.set_style('whitegrid')

from src.data.loader import load_validated

from src.features.instant import add_derived
from src.features.historical import add_label_free_aggregates

In [ ]:
df, _ = load_validated()
print('raw cols:', df.shape[1])

df2 = add_derived(df)
print('after instant FE:', df2.shape[1])

df3 = add_label_free_aggregates(df2)
print('after label-free historical FE:', df3.shape[1])

## Yeni feature'lar

In [ ]:
new_cols = [c for c in df3.columns if c not in df.columns]
print(new_cols)

print('\nÖrnek istatistikler:')
for c in ['device_tx_count_30d','receiver_tx_count_30d','device_distinct_accounts_all','receiver_first_seen_days_ago']:
    if c in df3.columns:
        print(f'  {c}: median={df3[c].median():.2f}, p99={df3[c].quantile(0.99):.2f}, max={df3[c].max():.2f}')

## Sinyal gücü — yeni feature'ların fraud ayrımındaki davranışı

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, col in zip(axes.flat, ['device_tx_count_30d','device_distinct_accounts_all','receiver_tx_count_30d','receiver_first_seen_days_ago']):
    if col not in df3.columns: continue
    for v, label in [(0,'non-fraud'),(1,'fraud')]:
        sub = np.log1p(df3.loc[df3['IsFraudTransaction']==v, col])
        sns.kdeplot(sub, ax=ax, label=label, fill=True, alpha=0.3)
    ax.set_title(col + ' (log1p)')
    ax.legend()
plt.tight_layout()

## Label-dependent feature'lar (lag varsayımıyla)

In [ ]:
from src.features.historical import add_label_dependent_aggregates
df4 = add_label_dependent_aggregates(df3, label_lag_days=7, prior_strength=50)
print('after label-dependent FE:', df4.shape[1])
print('device_fraud_rate_smoothed describe:')
print(df4['device_fraud_rate_smoothed'].describe(percentiles=[.5,.9,.99]).to_string())

## Sonuç
- Label-FREE set production'da güvenli — ilk turda kullanılacak.
- Label-DEPENDENT set, ≥7 gün lag varsayımı ile devreye alınabilir; ikinci tur ablation'la kazancı ölçülür.